# Week 2 Day 3 Lab - Durable State and Human in the Loop (free version)

## Pausing the agent for a handler's approval

UnderCover's rules say that a payout over $1,000 needs a handler's approval. A handler might not get to it for hours.

We rebuild the UnderCover agent as a LangGraph graph that saves its state in a Postgres database after every step. A payout over the limit pauses the graph until a handler approves it. If the process crashes, the claim carries on from where it stopped.

## 4 parts:

PART 1: Supabase

PART 2: From create_agent to a graph

PART 3: Pause for approval

PART 4: Crash and carry on

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of the lab. It uses <code>openrouter/free</code> throughout, so it needs no credit on OpenRouter. The full version with GPT-6 Luna is <a href="week2day3.ipynb">week2day3.ipynb</a>.<br/><br/>
            OpenRouter's free models allow 50 requests a day on an account without credit, and this lab makes about 15. <code>openrouter/free</code> picks a different free model for each request, so expect your results to vary more than mine, and some claims to take several minutes.<br/><br/>
            You'll need a free Supabase account, which we set up in Part 1. In Part 4 you restart the kernel on purpose, so read the box there before you run the cells after it.<br/><br/>
            Please execute each cell and compare your results with mine.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from urllib.parse import urlparse

import pandas as pd
import psycopg
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, RetryPolicy, interrupt
from psycopg.rows import dict_row
from pydantic import BaseModel, Field
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file in the `undercover` folder.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

## Part 1: Supabase

Supabase is a hosted Postgres database with a free plan. In this lab it holds the agent's state. In the product build it becomes UnderCover's database.

1. Sign up at [supabase.com](https://supabase.com). The free plan doesn't need a card.
2. Create a new project. Give it a database password of letters and numbers only, and keep a copy, as Supabase won't show it again. Pick the region closest to you.
3. When the project is ready, click **Connect** at the top of the page, then **Direct**, and choose **Session pooler**.
4. Copy the connection string and put your password in place of `[YOUR-PASSWORD]`.
5. Add it to the `.env` file in the `undercover` folder as `DATABASE_URL`.

We use the session pooler because it works over IPv4 and supports the prepared statements in LangGraph's checkpointer.

A free project pauses after a week without use. If yours has paused, restore it from the Supabase dashboard.

### Connecting

LangGraph's documentation asks for three settings on a checkpointer's connection: `autocommit=True`, `prepare_threshold=0` and `row_factory=dict_row`.

The lab's tables go in a schema of their own called `lab`, away from the tables UnderCover's product gets later today. `search_path` makes `lab` the schema for everything on this connection.

In [ ]:
load_dotenv(override=True)
DATABASE_URL = os.getenv("DATABASE_URL")

connection = psycopg.connect(DATABASE_URL, autocommit=True, prepare_threshold=0, row_factory=dict_row, options="-c search_path=lab")
connection.execute("CREATE SCHEMA IF NOT EXISTS lab")

print(connection.execute("SELECT version()").fetchone()["version"])

### The Supabase MCP server for Claude Code

In the product build later today, Claude Code looks at UnderCover's tables through Supabase's MCP server, read-only and limited to your project. This cell prints the command to add it. Run the command in a terminal in the `undercover` folder.

In [ ]:
project_ref = urlparse(DATABASE_URL).username.split(".")[1]

print(f'claude mcp add --transport http supabase "https://mcp.supabase.com/mcp?project_ref={project_ref}&read_only=true"')

Then run `/mcp` in Claude Code and choose Authenticate to sign in to Supabase in your browser.

## Part 2: From create_agent to a graph

### The claims

The agent works on claims from UnderCover's queue in `data/claims.json`. The approval limit is in `data/company.json`.

In [ ]:
DATA = Path("../data")


def load(name):
    """Read one of UnderCover's JSON files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


claims = {claim["claim_id"]: claim for claim in load("claims.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
APPROVAL_LIMIT = load("company.json")["payout_approval_limit"]

print(f"{len(claims)} claims. A payout over ${APPROVAL_LIMIT:,.0f} needs a handler's approval.")

### The agent we know

The instructions, the prompt and the payout tool are the ones from Day 2, with one change. The agent gives its decision in its reply, and the graph records it.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use the payout tool to work out the payout on an approved claim. When you have decided, give your decision, the payout and your reasons.
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [ ]:
@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

claims_agent = create_agent(model=llm, tools=[calculate_payout], system_prompt=f"{INSTRUCTIONS}\n{policy_wording}\n")

We run it on the passport eaten by a giraffe, as we did on Day 2. Free models vary, so if this cell fails, run it again.

In [ ]:
result = claims_agent.invoke({"messages": [{"role": "user", "content": make_user_prompt(claims["CLM-1005"])}]})
print(result["messages"][-1].text)

### create_agent builds a LangGraph graph

LangChain's agents are built on LangGraph. What `create_agent` returns is a compiled LangGraph graph.

In [ ]:
type(claims_agent)

LangGraph can describe its graph in Mermaid, a text format for diagrams. Here is the Mermaid for this one, and then the picture it draws.

In [ ]:
print(claims_agent.get_graph().draw_mermaid())

In [ ]:
display(Image(claims_agent.get_graph().draw_mermaid_png()))

It has a model node and a tools node, and it goes back and forth between them until the model stops asking for tools. Now we write a graph like this ourselves, with three more nodes.

### The same shape by hand

A LangGraph graph has three parts:

- **State:** what the graph keeps for one claim. The checkpointer saves it after every step.
- **Nodes:** Python functions that take the state and return the parts of it they change.
- **Edges:** which node runs next. A conditional edge is a function that picks the next node from the state.

`add_messages` adds a node's new messages to the list rather than replacing it.

In [ ]:
class ClaimState(TypedDict):
    """What the graph keeps for one claim."""

    messages: Annotated[list, add_messages]
    claim_id: str
    decision: str
    payout: float
    rationale: str
    approved_by: str

The agent node calls the model with the tools, as `create_agent` does. The decide node turns the agent's work into a structured decision, with the fields of the `ClaimDecision` from Week 1.

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

In [ ]:
model_with_tools = llm.bind_tools([calculate_payout])
decider = llm.with_structured_output(ClaimDecision)
system_message = SystemMessage(f"{INSTRUCTIONS}\n{policy_wording}\n")


def agent(state: ClaimState):
    """Call the model with the conversation so far."""
    return {"messages": [model_with_tools.invoke([system_message, *state["messages"]])]}


def decide(state: ClaimState):
    """Turn the agent's work into a decision."""
    result = decider.invoke([system_message, *state["messages"], HumanMessage("Record your decision on this claim.")])
    return {"decision": result.decision, "payout": result.payout, "rationale": result.rationale}

### Pausing for a handler

`interrupt()` pauses the graph. LangGraph saves the state along with the value we pass to `interrupt()`, and `invoke` returns. When we resume the graph with a handler's answer, `interrupt()` returns that answer and the node carries on.

In [ ]:
def approval(state: ClaimState):
    """Wait for a handler to approve a payout over the limit."""
    print(f"{state['claim_id']} needs a handler to approve ${state['payout']:,.2f}")
    answer = interrupt({"claim_id": state["claim_id"], "payout": state["payout"], "rationale": state["rationale"]})
    return {"approved_by": answer["handler"]}

### Paying the claim

Payments go in a table in Supabase. The claim id is the table's primary key, so a claim can only be paid once. The pay node changes nothing in the state, so it returns nothing.

In [ ]:
connection.execute("""
CREATE TABLE IF NOT EXISTS payments (
    claim_id TEXT PRIMARY KEY,
    amount NUMERIC NOT NULL,
    approved_by TEXT,
    paid_at TIMESTAMPTZ DEFAULT now()
)
""")


def pay(state: ClaimState):
    """Pay the claim, unless it has been paid already."""
    cursor = connection.execute(
        "INSERT INTO payments (claim_id, amount, approved_by) VALUES (%s, %s, %s) ON CONFLICT (claim_id) DO NOTHING",
        (state["claim_id"], state["payout"], state.get("approved_by")),
    )
    if cursor.rowcount:
        print(f"Paid {state['claim_id']} ${state['payout']:,.2f}")
    else:
        print(f"{state['claim_id']} was already paid")

### The edges

Two conditional edges choose the way through the graph.

In [ ]:
def after_agent(state: ClaimState):
    """Run the tools the model asked for, or move on to the decision."""
    if state["messages"][-1].tool_calls:
        return "tools"
    return "decide"


def after_decide(state: ClaimState):
    """Pay an approval, after a handler's approval if it's over the limit."""
    if state["decision"] != "approve":
        return END
    if state["payout"] > APPROVAL_LIMIT:
        return "approval"
    return "pay"

### The checkpointer

`PostgresSaver` saves the state in Supabase after every step. `setup()` creates its tables the first time you run it and does nothing after that.

In [ ]:
checkpointer = PostgresSaver(connection)
checkpointer.setup()

### Building the graph

The nodes and edges go into a `StateGraph`, and `compile` turns it into a graph we can run, with the checkpointer attached.

Free models sometimes send back a reply that can't be read. A `RetryPolicy` on a node runs it again when it fails, and `openrouter/free` picks another model for the next try.

In [ ]:
retry = RetryPolicy(max_attempts=3, retry_on=Exception)

builder = StateGraph(ClaimState)
builder.add_node("agent", agent, retry_policy=retry)
builder.add_node("tools", ToolNode([calculate_payout]))
builder.add_node("decide", decide, retry_policy=retry)
builder.add_node("approval", approval)
builder.add_node("pay", pay)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", after_agent, ["tools", "decide"])
builder.add_edge("tools", "agent")
builder.add_conditional_edges("decide", after_decide, ["approval", "pay", END])
builder.add_edge("approval", "pay")
builder.add_edge("pay", END)

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

Our agent node does the job of the model node in `create_agent`'s graph, so the loop with the tools is the same. The decide, approval and pay nodes are new.

## Part 3: Pause for approval

Each claim runs in its own thread, and the checkpointer keeps a thread's state under its id. We use the claim id as the thread id, so we can always find a claim's thread again.

`start_fresh` deletes a claim's thread and its payment, so you can run this lab more than once.

In [ ]:
def start_fresh(claim_id):
    """Delete a claim's saved thread and its payment."""
    checkpointer.delete_thread(claim_id)
    connection.execute("DELETE FROM payments WHERE claim_id = %s", (claim_id,))

### A claim under the limit

The passport eaten by a giraffe. With `durability="sync"`, LangGraph saves each step's checkpoint before it starts the next step.

In [ ]:
start_fresh("CLM-1005")
config = {"configurable": {"thread_id": "CLM-1005"}}
claim = {"messages": [HumanMessage(make_user_prompt(claims["CLM-1005"]))], "claim_id": "CLM-1005"}

result = graph.invoke(claim, config, durability="sync")
print(f"{result['decision']} ${result['payout']:,.2f}: {result['rationale']}")

### A claim over the limit

A fractured wrist in hospital abroad. The handler approved $3,750.

Free models vary. If the agent doesn't approve this claim, run the cell again for a fresh start.

In [ ]:
start_fresh("CLM-1003")
config = {"configurable": {"thread_id": "CLM-1003"}}
claim = {"messages": [HumanMessage(make_user_prompt(claims["CLM-1003"]))], "claim_id": "CLM-1003"}

result = graph.invoke(claim, config, durability="sync")
result["__interrupt__"]

The graph stopped at the approval node, and `invoke` returned what we passed to `interrupt()`. The state is saved in Supabase, and `next` says which node runs when the thread resumes.

In [ ]:
graph.get_state(config).next

### The handler approves

`Command(resume=...)` resumes the thread with the handler's answer.

In [ ]:
result = graph.invoke(Command(resume={"handler": "Dana Whitfield"}), config, durability="sync")

The approval node printed its message a second time. On resume, LangGraph runs the interrupted node again from the top, and this time `interrupt()` returns the handler's answer. Any code before the interrupt runs twice. That's why the payment has a node of its own.

In [ ]:
pd.DataFrame(connection.execute("SELECT * FROM payments ORDER BY paid_at").fetchall())

### Never pay twice

If a crash came in the middle of the pay node, LangGraph would run it again on resume. Because the claim id is the primary key, a second payment does nothing.

In [ ]:
pay(graph.get_state(config).values)

## Part 4: Crash and carry on

The farmhouse carried off by a tornado. This claim pauses for approval too.

Free models vary. If the agent doesn't approve this claim, run the cell again for a fresh start.

In [ ]:
start_fresh("CLM-1008")
config = {"configurable": {"thread_id": "CLM-1008"}}
claim = {"messages": [HumanMessage(make_user_prompt(claims["CLM-1008"]))], "claim_id": "CLM-1008"}

result = graph.invoke(claim, config, durability="sync")
result["__interrupt__"]

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Now the crash</h2>
            <span style="color:#EF476F;">Restart the kernel now. Everything in the notebook's memory is gone, including the graph and the connection.<br/><br/>
            Then run Parts 1 and 2 again to rebuild them. Select the first code cell of Part 3, the one that defines <code>start_fresh</code>, and choose <b>Execute Above Cells</b> from the menu beside the cell's run button in VS Code (or <b>Run All Above Selected Cell</b> from the Run menu in JupyterLab). Parts 1 and 2 only connect to Supabase and build the graph, so it's safe to run them again.<br/><br/>
            Then carry on from the cell after this box. Don't run Part 3 or the cell above this box again.
            </span>
        </td>
    </tr>
</table>

The new kernel has no memory of CLM-1008. Its thread is in Supabase, and we ask for it by its id.

In [ ]:
config = {"configurable": {"thread_id": "CLM-1008"}}
state = graph.get_state(config)

print(state.next)
print(state.interrupts[0].value)

The thread is waiting at the approval node, with everything the agent worked out. The handler approves, and the graph carries on.

In [ ]:
result = graph.invoke(Command(resume={"handler": "Priya Raman"}), config, durability="sync")

In [ ]:
pd.DataFrame(connection.execute("SELECT * FROM payments ORDER BY paid_at").fetchall())

### The checkpoints

`get_state_history` lists a thread's checkpoints, newest first, one for each step.

In [ ]:
for snapshot in graph.get_state_history(config):
    print(snapshot.metadata["step"], snapshot.next, snapshot.created_at)

In Supabase, open the **Table Editor**, choose the `lab` schema and look at the `checkpoints` table. It has a row for each step of each thread, with the thread id in the first column.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Most businesses have steps that need a person's approval: a large payment, a refund, a contract or a change to a customer's account. Durable state lets an agent stop at those steps and carry on when the approval comes, hours or days later.<br/><br/>
            Consider which steps in your business would need a person's approval if an agent did the work.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">The lab uses about 15 of the 50 free requests a day, so you may want to come back to these tomorrow.<br/><br/>
            Challenge: add a way for the handler to reject a payout. Let the handler's answer carry <code>approved</code> and a <code>note</code>, and when a handler rejects a payout, send the note back to the agent as a message so it can look at the claim again. Try it on CLM-1011, a lost suitcase where the payout is easy to get wrong. The handler paid $1,150.<br/><br/>
            Stretch challenge: on a second computer with the repo and the same <code>DATABASE_URL</code>, build the graph and approve a claim that you paused on the first one.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>